# Technical test — Transaction data analysis & graph investigation

**Context.** You join a compliance data team that detects suspicious money flows. You receive a (synthetic) extract of **~1,000 transactions**
between accounts. Before anyone builds a model, we need to know: *can we trust this data, what does it contain, and what does the network of accounts reveal?*

**Format** (~100-120 min; internet and docs allowed; pandas **or** Polars; think out loud — we value reasoning more than speed)
| Part | Topic | Time | |
|---|---|---|---|
| A | Data audit: is the data trustworthy? | ~35 min | core |
| B | Account-level features and simple rules | ~15 min | core |
| C | Graph analysis with NetworkX (hints given) | ~40 min | core |
| D | Open questions: an LLM / chatbot to help analysts | ~15 min | discussion + short code |

You do not need to finish everything. Explain assumptions in markdown cells. If you are stuck, ask — hints are fine.
`💡` marks a hint.

In [ ]:
import re, json
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
raw = pd.read_csv("transaction_data_test.csv")
raw.head()

## Part A — Data audit
Treat this like a real data-quality review: look first, then decide. Document every issue (what, how many, impact, your decision).

**A1. First look.** Shape, column types, memory, a few sample rows. Rename the columns to short snake_case names (keep a mapping). What does one row represent? Which columns are identifiers, measures, categories, timestamps, labels, or sensitive?

In [ ]:
# your code here

**A2. Completeness and uniqueness.** Missing values per column; duplicate rows; duplicate transaction IDs (careful: are two IDs the *same* ID written differently?); ID format checks (`TXN` + 10 digits, `ACC` + 5 digits).

💡 `df.duplicated()`, `Series.str.upper()`, `Series.str.match(regex)`

In [ ]:
# your code here

**A3. Validity.** Check the logic of each field: amount sign and range, sender = receiver, timestamp parsing and range (what period does the file cover?), allowed values of categorical columns, PIN format. Which rows are *impossible*, which are *suspicious but possible*?

In [ ]:
# your code here

**A4. Geolocation.** The column is a string like `"34.0522 N, -74.006 W"`. Parse it into numeric latitude/longitude and hemisphere. Does it make sense? (Think about hemisphere letters vs the sign of the number, how many distinct locations exist, and whether combinations look like real places.) Can you use it for a geographic risk feature?

💡 `Series.str.extract(r"...")`; `groupby(['lat','lon']).size()`

In [ ]:
# your code here

**A5. Distributions and outliers.** Describe the numeric columns. Plot amount (overall and by transaction type) and activity over time (per minute). Is `amount` plausibly real banking data? Detect outliers with a robust method (IQR or MAD) — then decide for each outlier group whether it is an *error* or a *signal*.

💡 `stats.kstest(x, 'uniform', args=(min, max-min))` tests how close a column is to uniform

In [ ]:
# your code here

**A6. The label.** `Fraud Flag` marks ~48% of transactions as fraud. Is that plausible? Does the flag relate to anything (amount, type, status, device, slice, latency, hour…)? Run suitable tests and conclude: can we train a supervised fraud model on this label?

💡 `pd.crosstab`, `stats.chi2_contingency`, `stats.mannwhitneyu`

In [ ]:
# your code here

**A7. Sensitive data & governance.** Which columns would you never let leave the secure environment, never log, and never paste into an external tool or LLM prompt? What do you do with them *now*?

In [ ]:
# your code here

**A8. Cleaning with an audit trail.** Write `clean(df)` returning `(clean_df, log)`. The log must say how many rows were removed or flagged per reason, and you must **not** silently drop anything. Decide what you drop, fix or only flag (self-transfers? out-of-hour timestamps? large amounts?). Justify in 3-4 lines.

In [ ]:
# your code here

## Part B — Account-level view
Analysts reason about **accounts**, not rows. Use the cleaned dataframe from A8 (`cdf`).

**B1. Account features.** Build one row per account (union of senders and receivers) with: number and total amount sent / received, number of **distinct** counterparties each way, first/last activity, and a pass-through ratio (amount out ÷ amount in, when both exist). Which accounts look unusual on each feature?

💡 `groupby().agg()` twice, then `outer` merge/join

In [ ]:
# your code here

**B2. Rules.** Write 3 simple, explainable rules (e.g. many distinct senders within an hour, many near-limit transfers, pass-through ≈ 1). For each: threshold, why that value (look at the distribution), and list the accounts flagged. How would you check false positives without a reliable label?

In [ ]:
# your code here

## Part C — Graph investigation with NetworkX
This is how we actually work on our AML pipeline. Hints are given; the goal is to interpret, not only to run functions.

**C1. Build the graph.** Create a directed graph (sender → receiver). One account pair can have several transactions: decide between `DiGraph` (aggregate: total amount, count, first/last timestamp) and `MultiDiGraph`, and justify. Store useful edge/node attributes.

💡 `nx.from_pandas_edgelist(..., create_using=nx.DiGraph, edge_attr=[...])` after a `groupby(['src','dst']).agg(...)`

In [ ]:
# your code here

**C2. Describe the network.** Number of nodes/edges, density, weakly connected components and their size distribution, in/out-degree distribution, share of nodes that both send and receive. What does this say about the *shape* of the data? (Would a typical graph algorithm — PageRank, community detection — be meaningful on the whole graph?)

💡 `nx.weakly_connected_components`, `nx.density`, `collections.Counter`

In [ ]:
# your code here

**C3. Find the unusual components.** Compute for each component: number of nodes/edges, whether it contains a directed cycle, max in-degree, max out-degree, and longest directed path. Which components are larger than what a purely random sparse graph would give (look at the size distribution)? Classify each unusual component by its topology.

💡 `G.subgraph(c)`, `nx.is_directed_acyclic_graph`, `nx.dag_longest_path_length`, `nx.find_cycle`

In [ ]:
# your code here

**C4. Characterise each pattern in AML terms** and check it with the data (use timestamps from `cdf`, not only the aggregated graph):
  a) *Circular flow* — list the accounts of the cycle, time span of the loop, how much value is lost around the loop.
  b) *Collector / pass-through* — the account with the highest in-degree: how much comes in, over how long, how much goes out and to whom (ratio?).
  c) *Structuring* — the sender with the highest out-degree: distribution of its amounts relative to the observed maximum amount, and the time window.
  d) *Layering chain* — follow the path: amounts per hop (is there a constant 'fee'?), time between hops (do the timestamps increase?).

💡 `nx.simple_cycles(G, length_bound=6)` (NetworkX >= 3.1), `nx.dag_longest_path`, `G.in_degree`, `G.successors`

In [ ]:
# your code here

**C5. Centrality & communities (and their limits).** Compute betweenness and in/out-degree centrality, and try a community-detection algorithm (e.g. Louvain) on the *union of the unusual components*. What do they add compared to C3-C4? Why would running them on the full graph be misleading here? What would change on a real dense network?

💡 `nx.betweenness_centrality`, `nx.community.louvain_communities(G.to_undirected(), seed=0)`

In [ ]:
# your code here

**C6. Compare with the `Fraud Flag`.** Take the accounts found in C3-C4. What share of their transactions carry `Fraud Flag = True`, compared with the base rate? What does this say about (a) the label and (b) your detection? Then produce a ranked list of the top accounts with **reason codes** an analyst could read.

In [ ]:
# your code here

**C7. (bonus) Draw it.** Plot the unusual components with node size ~ degree and edge width ~ amount; label the accounts. What would you show an analyst, and what would you hide?

In [ ]:
# your code here

## Part D — An LLM / chatbot for analysts: your ideas
No single right answer. We want structured thinking, awareness of risks, and a feel for what is realistic. Write 3-6 lines per question, then do the short coding exercise D6.

D1. **Use cases.** An analyst looks at an alert on one account. List 5 questions they would ask a chatbot. For each, say what data or computation is needed to answer it reliably (SQL, graph query, precomputed features, documents).

*Your answer:*




D2. **Architecture.** Compare: (a) the LLM writes SQL / graph queries from natural language, (b) the LLM only calls a fixed set of audited tools (e.g. `get_account_summary`, `find_cycles`), (c) the LLM reads documents (RAG). Which would you start with in a bank, and why? How do you prevent invented numbers?

*Your answer:*




D3. **Alert narratives.** Could an LLM draft the 'why this account is suspicious' text? What inputs would you give it, what must it never do, and how would you evaluate the text?

*Your answer:*




D4. **Privacy & compliance.** What can and cannot be sent to the LLM (think PIN, account ids, names, geolocation)? External API vs private deployment? What must be logged for audit?

*Your answer:*




D5. **Evaluation & limits.** How would you test the assistant before analysts use it (no reliable fraud labels)? What are its failure modes (wrong numbers, over-trust, prompt injection from documents), and when should it say 'I don't know'?

*Your answer:*




**D6. (code, short) Facts first, text second.** Write `case_facts(account)` returning a small JSON-serialisable dict of **verified facts** for one account (counts, totals, counterparties, time span, which rules from B2 fired). Then write `build_prompt(facts)` for an LLM that must write a 4-line analyst note using *only* those facts, and `check_numbers(note, facts)` that rejects a note containing a number not present in the facts. Test it on the collector account from C4 with a fake note.

In [ ]:
# your code here

## Wrap-up
In 5 lines: what are the 3 most important things you learned about this dataset, what would you ask the data owner, and what would you do next with one more day?

*Your answer:*
